# Kaggle submission: SR2 Qwen3-4B LoRA (merged) on T4 x2

Self-contained, offline inference on Python **3.13+** for the competition `test.csv`. It scores the
model produced by `model SR2-Qwen3-4B-lora --build`:

| Input        | Where                                                                  |
| ------------ | ---------------------------------------------------------------------- |
| `merged/`    | LoRA-merged `Qwen3Model` (BF16 weights), tokenizer and chat template   |
| `adapter.pt` | SR2 v3 comparison head (`head_state_dict`) plus compatibility metadata |

Pipeline: chat-template serialization, **no truncation / no `max_length`**, branch
de-duplication, length-sorted DP token-budget batches (longest first), a layer-split
FP16 backbone across two GPUs with asynchronous two-stage pipelining, pre-norm
last-real-token pooling, the FP32 SR2 head, and `submission.csv` validation.

Set `TEMPERATURE` at the very top of the first code cell and `MODEL_DIR` below
the startup diagnostics. Temperature must be finite and positive; `1.0` preserves
the original probabilities, values above `1` soften them, and values below `1`
sharpen them.

The upload workflow prepares dependencies before executing the notebook code:

```sh
pip install torch==2.14.0 torchvision==0.29.0 --index-url https://download.pytorch.org/whl/cu130
pip install transformers==5.17.0 pandas==3.0.5 safetensors==0.8.0 tqdm==4.70.1 packaging==26.3
```

The first code cell prints system, hardware, NVIDIA driver and dependency information after upload-time setup and before inference imports. It also uninstalls unused torchaudio from the kernel environment to avoid a mismatched CUDA audio extension. Runtime checks use project minimums and installed package constraints; lock-version differences are informational. Dependency installation belongs to upload-time setup; the inference cells use the prepared environment offline.


In [ ]:
# Temperature scaling: 1.0 preserves predictions; >1 softens, <1 sharpens.
TEMPERATURE = 1.0

import os

# Must be set before PyTorch / Hugging Face are imported; honor the legacy alias.
if "PYTORCH_CUDA_ALLOC_CONF" not in os.environ:
	os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

import inspect
import math
import platform
import shutil
import subprocess
import sys
from dataclasses import dataclass
from importlib import import_module, invalidate_caches, metadata
from pathlib import Path


class RuntimeDiagnostics:
	"""Print startup diagnostics and check the offline inference environment."""

	# Snapshot of uv.lock and the inference requirements in pyproject.toml.
	# Keep these embedded so Kaggle does not need a checkout of the project.
	LOCKED_VERSIONS = {
		"torch": "2.14.0+cu130",
		"transformers": "5.17.0",
		"numpy": "2.5.3",
		"pandas": "3.0.5",
		"safetensors": "0.8.0",
		"tokenizers": "0.23.2",
		"huggingface-hub": "1.31.0",
		"tqdm": "4.70.1",
		"packaging": "26.3",
	}
	PROJECT_REQUIREMENTS = {
		"pandas": "pandas>=3.0.5",
		"torch": "torch>=2.11.0",
		"transformers": "transformers>=5.16.1",
	}
	REQUIRED_APIS = {
		"torch": ("Tensor", "nn"),
		"numpy": (),
		"pandas": (),
		"transformers": ("AutoConfig", "AutoModel", "AutoTokenizer"),
		"transformers.integrations.sdpa_attention": ("use_gqa_in_sdpa",),
		"transformers.models.qwen3.modeling_qwen3": (
			"Qwen3DecoderLayer",
			"Qwen3RotaryEmbedding",
		),
		"safetensors": ("safe_open",),
		"torch.nn.functional": ("scaled_dot_product_attention",),
		"torch.nn.attention": (
			"sdpa_kernel",
			"SDPBackend.FLASH_ATTENTION",
			"SDPBackend.EFFICIENT_ATTENTION",
			"SDPBackend.CUDNN_ATTENTION",
		),
		"tqdm.auto": ("tqdm",),
	}

	@staticmethod
	def _report(label, reader) -> None:
		"""An optional diagnostic must not hide the rest of the report."""
		try:
			value = reader()
		except Exception as error:
			value = f"unavailable ({type(error).__name__}: {error})"
		print(f"{label}: {value}", flush=True)

	@staticmethod
	def _cpu_model() -> str:
		if sys.platform.startswith("linux"):
			for line in Path("/proc/cpuinfo").read_text().splitlines():
				key, separator, value = line.partition(":")
				if separator and key.strip() in {"model name", "Hardware"}:
					return value.strip()
		return platform.processor() or platform.machine()

	@staticmethod
	def _memory() -> str:
		if sys.platform.startswith("linux"):
			values = {}
			for line in Path("/proc/meminfo").read_text().splitlines():
				key, value = line.split(":", 1)
				values[key] = int(value.split()[0]) * 1024
			return (
				f"{values['MemTotal'] / 2**30:.1f} GiB total, "
				f"{values['MemAvailable'] / 2**30:.1f} GiB available"
			)
		if sys.platform == "win32":
			import ctypes

			class MemoryStatus(ctypes.Structure):
				_fields_ = [
					("length", ctypes.c_ulong),
					("load", ctypes.c_ulong),
				] + [
					(name, ctypes.c_ulonglong)
					for name in (
						"total_phys",
						"avail_phys",
						"total_page",
						"avail_page",
						"total_virtual",
						"avail_virtual",
						"avail_extended",
					)
				]

			status = MemoryStatus()
			status.length = ctypes.sizeof(status)
			if not ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(status)):
				raise ctypes.WinError()
			return (
				f"{status.total_phys / 2**30:.1f} GiB total, "
				f"{status.avail_phys / 2**30:.1f} GiB available"
			)
		total = os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE")
		return f"{total / 2**30:.1f} GiB total"

	@staticmethod
	def _disk() -> str:
		usage = shutil.disk_usage(Path.cwd())
		return f"{usage.total / 2**30:.1f} GiB total, {usage.free / 2**30:.1f} GiB free"

	@staticmethod
	def _command(name: str, *arguments: str) -> None:
		print(f"\n--- {name} ---", flush=True)
		executable = shutil.which(name)
		if executable is None:
			print("Unavailable: executable not found on PATH.", flush=True)
			return
		try:
			result = subprocess.run(
				[executable, *arguments],
				capture_output=True,
				text=True,
				encoding="utf-8",
				errors="replace",
				timeout=10,
				check=False,
			)
		except (OSError, subprocess.TimeoutExpired) as error:
			print(f"Unavailable: {type(error).__name__}: {error}", flush=True)
			return
		print(result.stdout.strip() or "(no stdout)", flush=True)
		if result.stderr.strip():
			print(result.stderr.strip(), flush=True)
		if result.returncode:
			print(f"Exit status: {result.returncode}", flush=True)

	def report_system(self) -> None:
		print("=== System and hardware (before inference imports) ===", flush=True)
		self._report("Operating system", platform.platform)
		self._report("Kernel", lambda: f"{platform.system()} {platform.release()}")
		self._report("Architecture", platform.machine)
		self._report("Python", lambda: sys.version.replace("\n", " "))
		self._report("Interpreter", lambda: sys.executable)
		self._report("CPU", self._cpu_model)
		self._report("Logical CPU cores", os.cpu_count)
		self._report("System memory", self._memory)
		self._report("Working directory", Path.cwd)
		self._report("Working-directory disk", self._disk)
		print(
			"\nNVIDIA-SMI CUDA = driver-supported CUDA; nvcc = installed Toolkit.\n"
			"PyTorch's CUDA build is reported separately after import. "
			"nvcc is optional for prebuilt PyTorch wheels.",
			flush=True,
		)
		self._command("nvidia-smi")
		self._command("nvcc", "--version")

	@staticmethod
	def remove_unused_torchaudio() -> None:
		"""Remove the unused audio extension before PyTorch/Transformers imports."""
		print("\n=== Remove unused TorchAudio (offline) ===", flush=True)
		try:
			version = metadata.version("torchaudio")
		except metadata.PackageNotFoundError:
			print("torchaudio is not installed; skipping.", flush=True)
			return
		print(
			f"Uninstalling torchaudio {version} from {sys.executable} to avoid "
			"a CUDA build conflict with PyTorch.",
			flush=True,
		)
		try:
			result = subprocess.run(
				[
					sys.executable,
					"-m",
					"pip",
					"--disable-pip-version-check",
					"uninstall",
					"-y",
					"torchaudio",
				],
				capture_output=True,
				text=True,
				timeout=120,
				check=True,
			)
		except subprocess.CalledProcessError as error:
			details = (error.stderr or error.stdout or str(error)).strip()
			raise RuntimeError(
				f"Could not uninstall unused torchaudio before inference imports:\n{details}"
			) from error
		except (OSError, subprocess.TimeoutExpired) as error:
			raise RuntimeError(
				f"Could not uninstall unused torchaudio before inference imports: {error}"
			) from error
		for output in (result.stdout, result.stderr):
			if output.strip():
				print(output.strip(), flush=True)
		invalidate_caches()
		print("Unused torchaudio removed before inference imports.", flush=True)

	def report_dependencies(self) -> None:
		print("\n=== Inference dependencies (after upload-time setup) ===", flush=True)
		print(
			f"{'Package':<18} {'Installed':<24} {'uv.lock':<20} Project minimum",
			flush=True,
		)
		for name, locked in self.LOCKED_VERSIONS.items():
			try:
				installed = metadata.version(name)
			except metadata.PackageNotFoundError:
				installed = "MISSING"
			except Exception as error:
				installed = f"unavailable ({type(error).__name__})"
			minimum = self.PROJECT_REQUIREMENTS.get(name, "-")
			note = "" if installed == locked else " [differs from lock]"
			print(
				f"{name:<18} {installed:<24} {locked:<20} {minimum}{note}", flush=True
			)
		print(
			"Lock differences, including CUDA build tags, are informational. "
			"Project minimums, installed base dependency constraints and required APIs "
			"are checked before model loading.",
			flush=True,
		)

	@staticmethod
	def _stop(errors: list[str]) -> None:
		if errors:
			details = "\n- ".join(dict.fromkeys(errors))
			raise RuntimeError(
				f"Offline inference preflight failed:\n- {details}\n"
				"Prepare compatible inference dependencies in this kernel environment "
				"before running; restart the kernel after replacing imported packages. "
				"Upload-time dependency setup must complete before inference starts."
			)

	def check_dependencies(self) -> None:
		"""Check installed base runtime dependencies, without optional extras."""
		errors = []
		if sys.version_info < (3, 13):
			errors.append("The standalone notebook requires Python >=3.13.")
		try:
			from packaging.requirements import Requirement
			from packaging.specifiers import SpecifierSet
			from packaging.utils import canonicalize_name
			from packaging.version import Version
		except Exception as error:
			errors.append(f"Cannot import packaging for version checks: {error}")
			self._stop(errors)

		python_version = Version(platform.python_version())
		pending = [(name, "notebook") for name in self.LOCKED_VERSIONS]
		pending.extend(
			(requirement, "pyproject.toml")
			for requirement in self.PROJECT_REQUIREMENTS.values()
		)
		visited = set()
		while pending:
			raw, parent = pending.pop()
			try:
				requirement = Requirement(raw)
				if requirement.marker and not requirement.marker.evaluate(
					{"extra": ""}
				):
					continue
				distribution = metadata.distribution(requirement.name)
				installed = Version(distribution.version)
				if not requirement.specifier.contains(installed):
					errors.append(
						f"{parent} requires {requirement}; installed "
						f"{requirement.name}=={installed}."
					)
				name = canonicalize_name(requirement.name)
				if name in visited:
					continue
				visited.add(name)
				requires_python = distribution.metadata.get("Requires-Python")
				if requires_python and not SpecifierSet(requires_python).contains(
					python_version
				):
					errors.append(
						f"{requirement.name}=={installed} requires Python "
						f"{requires_python}; running {python_version}."
					)
				pending.extend(
					(dependency, requirement.name)
					for dependency in distribution.requires or ()
				)
			except metadata.PackageNotFoundError:
				errors.append(f"{parent} requires {raw}; the package is missing.")
			except Exception as error:
				errors.append(f"Cannot check {raw} required by {parent}: {error}")
		self._stop(errors)
		print(
			"Project minimums and installed base dependency constraints: OK", flush=True
		)

	def _report_torch(self, torch_module, errors: list[str]) -> None:
		print("\n=== PyTorch CUDA runtime ===", flush=True)
		self._report("PyTorch", lambda: torch_module.__version__)
		self._report("PyTorch CUDA build", lambda: torch_module.version.cuda)
		self._report("cuDNN version", lambda: torch_module.backends.cudnn.version())
		self._report(
			"Compiled GPU architectures", lambda: torch_module.cuda.get_arch_list()
		)
		try:
			available = torch_module.cuda.is_available()
			count = torch_module.cuda.device_count()
		except Exception as error:
			errors.append(f"Cannot initialize CUDA: {error}")
			return
		print(f"CUDA available: {available} | visible GPUs: {count}", flush=True)
		if not available or count == 0:
			errors.append(
				"CUDA is unavailable. Select a GPU runtime and prepare a CUDA-enabled "
				"PyTorch build compatible with the NVIDIA driver."
			)
			return
		for index in range(count):

			def device_info(index=index):
				properties = torch_module.cuda.get_device_properties(index)
				return (
					f"{properties.name} | compute capability "
					f"{properties.major}.{properties.minor} | "
					f"{properties.total_memory / 2**30:.1f} GiB total"
				)

			def free_memory(index=index):
				free, total = torch_module.cuda.mem_get_info(index)
				return f"{free / 2**30:.1f} / {total / 2**30:.1f} GiB free / total"

			self._report(f"cuda:{index}", device_info)
			self._report(f"cuda:{index} memory", free_memory)

	def check_runtime(self) -> None:
		"""Import inference modules and check the SDPA/Qwen3 interfaces in use."""
		errors = []
		modules = {}
		for name, attributes in self.REQUIRED_APIS.items():
			try:
				module = import_module(name)
				modules[name] = module
			except Exception as error:
				errors.append(f"Cannot import {name}: {type(error).__name__}: {error}")
				continue
			for attribute in attributes:
				try:
					value = module
					for part in attribute.split("."):
						value = getattr(value, part)
				except Exception as error:
					errors.append(
						f"Required API {name}.{attribute} is unavailable: "
						f"{type(error).__name__}: {error}"
					)

		qwen = modules.get("transformers.models.qwen3.modeling_qwen3")
		if qwen is not None:
			try:
				parameters = inspect.signature(
					qwen.Qwen3DecoderLayer.forward
				).parameters
				required = {
					"attention_mask",
					"position_ids",
					"position_embeddings",
					"past_key_values",
					"use_cache",
				}
				missing = required - parameters.keys()
				if missing:
					errors.append(
						f"Qwen3DecoderLayer.forward lacks parameters: {sorted(missing)}."
					)
			except (AttributeError, TypeError, ValueError) as error:
				errors.append(f"Cannot inspect the Qwen3 decoder API: {error}")
		if "torch" in modules:
			self._report_torch(modules["torch"], errors)
		self._stop(errors)
		print(
			"Inference imports, required SDPA/Qwen3 APIs and CUDA availability: OK",
			flush=True,
		)


DIAGNOSTICS = RuntimeDiagnostics()
DIAGNOSTICS.report_system()
DIAGNOSTICS.remove_unused_torchaudio()
DIAGNOSTICS.report_dependencies()

# Kaggle model input directory holding `merged/` and `adapter.pt` (any nesting
# depth). If it does not exist, /kaggle/input is searched for `adapter.pt` instead.
MODEL_DIR = Path(
	"/kaggle/input/models/dnbide/sr2-qwen3-4b-lora/transformers/default/1/SR2_Qwen3_4B_lora"
)


@dataclass(frozen=True, slots=True)
class SubmitConfig:
	"""Every tunable of the notebook in one place."""

	temperature: float = TEMPERATURE
	model_dir: Path | None = MODEL_DIR
	test_csv: Path = Path(
		"/kaggle/input/competitions/llm-classification-finetuning/test.csv"
	)
	sample_submission: Path = Path(
		"/kaggle/input/competitions/llm-classification-finetuning/sample_submission.csv"
	)
	output_path: Path = Path("/kaggle/working/submission.csv")
	# Only used when the dataset's shard names differ from the index.
	staging_dir: Path = Path("/kaggle/working/model_links")
	# Stage 0 (embeddings + first layers) and stage 1 (last layers + norm) devices.
	# With a single visible GPU both stages share it and nothing is split.
	devices: tuple[str, ...] = ("cuda:0", "cuda:1")
	# First layer of stage 1; None splits the layers evenly (18 / 18).
	split_layer: int | None = None
	# T4 has no BF16 tensor cores: the BF16 weights are cast to FP16.
	precision: str = "fp16"
	# Padded tokens (rows * longest row) per backbone batch.
	token_budget: int = 32768
	max_rows_per_batch: int = 128
	# Batches in flight; 2 lets GPU0 start batch k+1 while GPU1 finishes batch k.
	pipeline_depth: int = 2
	# "off" makes Transformers repeat K/V heads (use_gqa_in_sdpa -> False), which keeps
	# the memory-efficient kernel eligible on T4. "on" forces native
	# enable_gqa; "auto" probes it on the device and keeps it only if it is faster.
	use_gqa: str = "off"
	# Training class prior [A, B, tie] for rows whose features are not finite.
	class_prior: tuple[float, float, float] = (0.349, 0.342, 0.309)

	def __post_init__(self) -> None:
		if not math.isfinite(self.temperature) or self.temperature <= 0:
			raise ValueError("temperature must be a finite positive number.")


CONFIG = SubmitConfig()

## Inference environment preflight

Requires Python 3.13+ and the Python versions supported by the installed inference dependencies. The full project's Python 3.14 requirement does not apply to this standalone notebook. Dependency setup runs during upload, before notebook execution; all preflight checks run against the prepared environment before model loading.


In [ ]:
DIAGNOSTICS.check_dependencies()
DIAGNOSTICS.check_runtime()

import gc
import hashlib
import json
import time
from collections import deque
from contextlib import AbstractContextManager, nullcontext

import numpy as np
import pandas as pd
import torch
import transformers.integrations.sdpa_attention as sdpa_attention
from safetensors import safe_open
from torch import Tensor, nn
from torch.nn import functional as F
from torch.nn.attention import SDPBackend, sdpa_kernel
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoModel, AutoTokenizer

## Model artifacts and the SR2 v3 head

The head is a verbatim port of `SwapEquivariantHarmonicDiscrepancyClassificationHead`
(`models/SR2_Qwen3_4B_lora/heads.py`) in evaluation mode. Features arrive in FP32, so
the whole head runs in FP32 and returns FP32 logits `[A-win, B-win, tie]`.


In [ ]:
HEAD_ARCHITECTURE = "swap_equivariant_harmonic_discrepancy_v3"
CLASS_COLUMNS = ("winner_model_a", "winner_model_b", "winner_tie")


@dataclass(frozen=True, slots=True)
class ModelArtifacts:
	"""Locations of the merged backbone directory and the adapter/head checkpoint."""

	merged_dir: Path
	adapter_path: Path

	@classmethod
	def discover(cls, model_dir: Path | None, staging_dir: Path) -> "ModelArtifacts":
		"""Find `adapter.pt` next to a `merged/` directory and normalize shard names."""
		if model_dir is not None and model_dir.exists():
			root = model_dir
		else:
			root = Path("/kaggle/input")
			if model_dir is not None:
				print(f"MODEL_DIR {model_dir} does not exist; searching {root}.")
		for adapter in sorted(root.rglob("adapter.pt")):
			merged = adapter.parent / "merged"
			if (merged / "config.json").is_file():
				return cls(cls._normalize_shards(merged, staging_dir), adapter)
		raise FileNotFoundError(
			f"No directory with adapter.pt and merged/config.json under {root}."
		)

	@staticmethod
	def _expected_shards(merged: Path) -> list[str]:
		index_path = merged / "model.safetensors.index.json"
		if index_path.is_file():
			weight_map = json.loads(index_path.read_text(encoding="utf-8"))
			return sorted(set(weight_map["weight_map"].values()))
		return ["model.safetensors"]

	@classmethod
	def _normalize_shards(cls, merged: Path, staging_dir: Path) -> Path:
		"""Symlink shards named like `model-00001-of-00002-001.safetensors` to index names."""
		expected = cls._expected_shards(merged)
		if all((merged / name).is_file() for name in expected):
			return merged
		renames: dict[str, str] = {}
		for name in expected:
			if (merged / name).is_file():
				continue
			candidates = sorted(merged.glob(f"{Path(name).stem}-*.safetensors"))
			if len(candidates) != 1:
				raise FileNotFoundError(
					f"Shard {name} is missing from {merged} "
					f"(suffixed candidates: {[c.name for c in candidates]})."
				)
			renames[candidates[0].name] = name
		if staging_dir.exists():
			shutil.rmtree(staging_dir)
		staging_dir.mkdir(parents=True)
		for path in merged.iterdir():
			(staging_dir / renames.get(path.name, path.name)).symlink_to(path)
		print(f"Linked {len(renames)} renamed shard(s) into {staging_dir}.")
		return staging_dir


class SwapEquivariantHead(nn.Module):
	"""SR2 v3 head: cross-projected wins, harmonic tie fusion, discrepancy correction."""

	EPSILON = 1e-8

	def __init__(self, backbone_hidden_size: int, hidden_size: int) -> None:
		super().__init__()
		self.input_projection = nn.Linear(
			backbone_hidden_size, 2 * hidden_size, bias=False
		)
		self.win_projection_bias = nn.Parameter(torch.zeros(hidden_size))
		self.tie_projection_bias = nn.Parameter(torch.zeros(hidden_size))
		self.discrepancy_projection = nn.Linear(hidden_size, hidden_size, bias=True)
		self.win_readout = nn.Linear(hidden_size, 1, bias=False)
		self.tie_readout = nn.Linear(hidden_size, 1, bias=False)
		self.discrepancy_readout = nn.Linear(hidden_size, 1, bias=False)
		self.tie_bias = nn.Parameter(torch.zeros(()))
		self.activation = nn.GELU()

	def forward(self, h_a: Tensor, h_b: Tensor) -> Tensor:
		"""Return FP32 `[A-win, B-win, tie]` logits for original-orientation features."""
		branches = torch.stack((h_a, h_b), dim=1)
		self_features, cross_features = self.input_projection(branches).chunk(2, dim=-1)
		wins = self.activation(
			self_features + cross_features.flip(1) + self.win_projection_bias
		)
		l_weight, m_weight = self.input_projection.weight.chunk(2, dim=0)
		tie_features = F.linear(branches, l_weight - m_weight, self.tie_projection_bias)
		x, y = tie_features.unbind(dim=1)
		fused = 2 * x * y / (x.abs() + y.abs() + self.EPSILON)
		discrepancy = self.activation(self.discrepancy_projection((x - y).abs()))
		win_logits = self.win_readout(wins).squeeze(-1)
		s = self.tie_readout(fused)
		q = self.discrepancy_readout(discrepancy)
		return torch.cat((win_logits + q, s - q + self.tie_bias), dim=-1)


class HeadLoader:
	"""Validate checkpoint/export compatibility and restore the FP32 head."""

	REQUIRED_METADATA = {
		"schema_version": 2,
		"class_count": 3,
		"pooling": "last-nonpadding-token",
		"head_architecture": HEAD_ARCHITECTURE,
	}

	def __init__(self, artifacts: ModelArtifacts) -> None:
		self._artifacts = artifacts

	def load(self, backbone_hidden_size: int) -> SwapEquivariantHead:
		checkpoint = self._read_checkpoint()
		metadata = checkpoint["metadata"]
		for name, expected in self.REQUIRED_METADATA.items():
			if metadata.get(name) != expected:
				raise ValueError(
					f"adapter.pt metadata {name}={metadata.get(name)!r}, expected {expected!r}."
				)
		if metadata["backbone_hidden_size"] != backbone_hidden_size:
			raise ValueError("adapter.pt and merged backbone hidden sizes differ.")
		self._check_export(metadata)
		head = SwapEquivariantHead(
			backbone_hidden_size, int(metadata["classifier_hidden_size"])
		)
		head.load_state_dict(
			{
				k: v.detach().clone().float()
				for k, v in checkpoint["head_state_dict"].items()
			},
			strict=True,
		)
		print(
			f"Head: {metadata['head_architecture']} h={metadata['classifier_hidden_size']} "
			f"| trained {metadata.get('created_at')} | base {metadata.get('base_revision')}"
		)
		del checkpoint
		return head.eval().requires_grad_(False)

	def _read_checkpoint(self) -> dict:
		path = self._artifacts.adapter_path
		try:
			checkpoint = torch.load(
				path, map_location="cpu", weights_only=True, mmap=True
			)
		except (RuntimeError, ValueError):
			checkpoint = torch.load(path, map_location="cpu", weights_only=True)
		if not isinstance(checkpoint.get("metadata"), dict) or not isinstance(
			checkpoint.get("head_state_dict"), dict
		):
			raise ValueError("adapter.pt lacks metadata or head_state_dict.")
		return checkpoint

	def _check_export(self, metadata: dict) -> None:
		"""Reject a merged backbone that was exported from a different checkpoint."""
		path = self._artifacts.merged_dir / "export_metadata.json"
		if not path.is_file():
			print(
				"export_metadata.json missing; skipping the backbone/head pairing check."
			)
			return
		export = json.loads(path.read_text(encoding="utf-8"))
		with self._artifacts.adapter_path.open("rb") as file:
			digest = hashlib.file_digest(file, "sha256").hexdigest()
		if export.get("checkpoint_sha256") != digest:
			raise ValueError(
				"merged backbone was exported from a different adapter.pt."
			)
		if export.get("base_revision") != metadata.get("base_revision"):
			raise ValueError(
				"merged backbone and adapter.pt use different base revisions."
			)

## Conversation serialization and tokenization

Same messages as training: alternating user/assistant turns, the tokenizer's own chat
template, no system message, no generation prompt, `add_special_tokens=False`.
Nothing is truncated. Identical token sequences are encoded once.


In [ ]:
@dataclass(slots=True)
class BranchCorpus:
	"""Unique tokenized branches plus per-row indices of the A and B branch."""

	tokens: list[np.ndarray]
	a_index: np.ndarray
	b_index: np.ndarray
	pad_token_id: int

	@property
	def lengths(self) -> np.ndarray:
		return np.fromiter((len(t) for t in self.tokens), dtype=np.int64)


class ConversationSerializer:
	"""Render one branch with the tokenizer's official chat template."""

	def __init__(self, tokenizer) -> None:
		if not tokenizer.chat_template:
			raise ValueError("The merged tokenizer has no chat template.")
		self._tokenizer = tokenizer
		self.lenient_repairs = 0

	def branch(self, prompt, response) -> str:
		prompts, responses = self._turns(prompt), self._turns(response)
		if not prompts or len(prompts) != len(responses):
			self.lenient_repairs += 1
			count = max(1, len(prompts), len(responses))
			prompts = (prompts + [""] * count)[:count]
			responses = (responses + [""] * count)[:count]
		messages: list[dict[str, str]] = []
		for user, assistant in zip(prompts, responses, strict=True):
			messages.append({"role": "user", "content": user})
			messages.append({"role": "assistant", "content": assistant})
		return self._tokenizer.apply_chat_template(
			messages,
			tokenize=False,
			add_generation_prompt=False,
			continue_final_message=False,
		)

	@staticmethod
	def _turns(value) -> list[str]:
		if value is None or (isinstance(value, float) and math.isnan(value)):
			return []
		parsed = value
		if isinstance(value, str):
			try:
				parsed = json.loads(value)
			except json.JSONDecodeError:
				parsed = [value]
		if not isinstance(parsed, list):
			parsed = [parsed]
		return ["" if turn is None else str(turn) for turn in parsed]


class BranchTokenizer:
	"""Serialize, encode without truncation, and de-duplicate all branches."""

	CHUNK = 512

	def __init__(self, tokenizer) -> None:
		self._tokenizer = tokenizer
		self._serializer = ConversationSerializer(tokenizer)
		# Silences the length warning only; truncation stays disabled below.
		tokenizer.model_max_length = int(1e9)

	def build(self, frame: pd.DataFrame) -> BranchCorpus:
		started = time.perf_counter()
		unique: dict[bytes, int] = {}
		tokens: list[np.ndarray] = []
		a_index = np.empty(len(frame), dtype=np.int64)
		b_index = np.empty(len(frame), dtype=np.int64)
		rows = list(
			zip(
				frame["prompt"],
				frame["response_a"],
				frame["response_b"],
				strict=True,
			)
		)
		for start in tqdm(range(0, len(rows), self.CHUNK), desc="Tokenizing"):
			chunk = rows[start : start + self.CHUNK]
			texts = [
				self._serializer.branch(prompt, response)
				for prompt, response_a, response_b in chunk
				for response in (response_a, response_b)
			]
			encoded = self._tokenizer(
				texts,
				add_special_tokens=False,
				truncation=False,
				padding=False,
				return_attention_mask=False,
			)["input_ids"]
			for offset, ids in enumerate(encoded):
				array = np.asarray(ids, dtype=np.int32)
				key = hashlib.blake2b(array.tobytes(), digest_size=16).digest()
				position = unique.setdefault(key, len(tokens))
				if position == len(tokens):
					tokens.append(array)
				row = start + offset // 2
				(a_index if offset % 2 == 0 else b_index)[row] = position
		pad = self._tokenizer.pad_token_id
		corpus = BranchCorpus(tokens, a_index, b_index, 0 if pad is None else int(pad))
		lengths = corpus.lengths
		print(
			f"{len(frame):,} rows -> {2 * len(frame):,} branches, {len(tokens):,} unique | "
			f"tokens min/median/max {lengths.min():,}/{int(np.median(lengths)):,}/{lengths.max():,} "
			f"| total {int(lengths.sum()):,} | no truncation | "
			f"{self._serializer.lenient_repairs} repaired rows | "
			f"{time.perf_counter() - started:.0f}s"
		)
		return corpus

## Attention policy, split backbone and DP token-budget batching

- `AttentionPolicy` patches `use_gqa_in_sdpa` (default: native GQA disabled, so K/V heads
  are repeated and the memory-efficient kernel stays eligible on T4)
  and allows only fused kernels, so a silent quadratic math fallback is impossible.
  `use_gqa="auto"` probes native `enable_gqa` and adopts it only if it works and is faster.
- `SplitBackbone` streams safetensors shards directly onto GPU0 (`embed_tokens` + first
  half of layers) and GPU1 (remaining layers + `norm`) without materializing the 8 GB
  model in host RAM or installing accelerate hooks. Only the intermediate hidden tensor
  crosses PCIe at the split boundary, and the last real token is pooled **before**
  `Qwen3RMSNorm` to avoid allocating a full-sequence FP32 norm buffer.
- `TokenBudgetPlanner` sorts unique branches longest first and chooses the better of
  descending best-fit and optimal contiguous dynamic programming on
  `(batch_count, padded_tokens)` under `token_budget` and `max_rows_per_batch`.


In [ ]:
class AttentionPolicy(AbstractContextManager):
	"""Select and patch GQA behaviour, and scope fused-only SDPA backends."""

	FUSED_BACKENDS = (
		SDPBackend.FLASH_ATTENTION,
		SDPBackend.EFFICIENT_ATTENTION,
		SDPBackend.CUDNN_ATTENTION,
	)

	def __init__(self, mode: str, devices: list[torch.device]) -> None:
		if mode not in {"auto", "on", "off"}:
			raise ValueError("use_gqa must be 'auto', 'on' or 'off'.")
		self._mode = mode
		self._devices = devices
		self._fused_only = all(d.type == "cuda" for d in devices)
		self._original = sdpa_attention.use_gqa_in_sdpa
		self.use_gqa = True

	def select(self) -> bool:
		"""Decide native GQA vs repeat_kv; probes the first CUDA device in auto mode."""
		if self._mode != "auto" or not self._fused_only:
			self.use_gqa = self._mode != "off"
		else:
			self.use_gqa = self._probe(self._devices[0])
		print(f"SDPA GQA path: {'native enable_gqa' if self.use_gqa else 'repeat_kv'}")
		return self.use_gqa

	def __enter__(self) -> "AttentionPolicy":
		use_gqa = self.use_gqa
		# Version-agnostic: the signature differs between transformers releases.
		sdpa_attention.use_gqa_in_sdpa = lambda *args, **kwargs: use_gqa
		return self

	def __exit__(self, *exc_info) -> None:
		sdpa_attention.use_gqa_in_sdpa = self._original

	def kernels(self) -> AbstractContextManager:
		"""Fused kernels only (Flash / memory-efficient / cuDNN); no math fallback."""
		if not self._fused_only:
			return nullcontext()
		return sdpa_kernel(list(self.FUSED_BACKENDS))

	def _probe(self, device: torch.device, length: int = 8192) -> bool:
		generator = torch.Generator(device=device).manual_seed(0)
		q = torch.randn(
			1, 32, length, 128, device=device, dtype=torch.float16, generator=generator
		)
		k = torch.randn(
			1, 8, length, 128, device=device, dtype=torch.float16, generator=generator
		)
		v = torch.randn(
			1, 8, length, 128, device=device, dtype=torch.float16, generator=generator
		)

		def run(native: bool):
			torch.cuda.synchronize(device)
			torch.cuda.reset_peak_memory_stats(device)
			base = torch.cuda.memory_allocated(device)
			started = time.perf_counter()
			for _ in range(3):
				kk, vv = (
					(k, v)
					if native
					else (k.repeat_interleave(4, 1), v.repeat_interleave(4, 1))
				)
				out = F.scaled_dot_product_attention(
					q, kk, vv, is_causal=True, enable_gqa=native
				)
				del kk, vv
			torch.cuda.synchronize(device)
			elapsed = (time.perf_counter() - started) / 3
			return out, elapsed, torch.cuda.max_memory_allocated(device) - base

		try:
			with self.kernels():
				run(True)  # warm-up
				run(False)
				native_out, native_time, native_peak = run(True)
				repeat_out, repeat_time, repeat_peak = run(False)
		except RuntimeError as error:
			print(
				f"Native GQA is unavailable on {device} ({str(error).splitlines()[0]})."
			)
			return False
		finally:
			torch.cuda.empty_cache()
		close = (native_out - repeat_out).abs().max().item() < 5e-2
		print(
			f"GQA probe L={length}: native {native_time * 1e3:.1f} ms / {native_peak / 2**20:.0f} MiB, "
			f"repeat_kv {repeat_time * 1e3:.1f} ms / {repeat_peak / 2**20:.0f} MiB, match={close}"
		)
		return (
			close
			and native_time <= 1.1 * repeat_time
			and native_peak <= 1.05 * repeat_peak
		)


class SplitBackbone:
	"""Merged Qwen3 backbone streamed directly onto two stage devices."""

	DTYPES = {"fp16": torch.float16, "bf16": torch.bfloat16, "fp32": torch.float32}

	def __init__(
		self,
		merged_dir: Path,
		devices: list[torch.device],
		split_layer: int | None,
		precision: str,
	) -> None:
		started = time.perf_counter()
		dtype = self.DTYPES[precision]
		config = AutoConfig.from_pretrained(merged_dir)
		if "sliding_attention" in getattr(config, "layer_types", ()):
			raise ValueError(
				"Sliding-window layers are not supported by mask-free causal encoding."
			)
		config.use_cache = False
		layer_count = int(config.num_hidden_layers)
		first, last = devices[0], devices[-1]
		split = (
			layer_count
			if first == last
			else (layer_count // 2 if split_layer is None else split_layer)
		)
		if not 0 < split <= layer_count:
			raise ValueError(f"split_layer must be in 1..{layer_count}.")
		norm_device = first if split == layer_count else last

		with torch.device("meta"):
			model = AutoModel.from_config(config, attn_implementation="sdpa")
		self._stream_weights(model, merged_dir, dtype, first, last, norm_device, split)
		self._rotary_first = type(model.rotary_emb)(config).to(first)
		self._rotary_last = (
			self._rotary_first
			if first == norm_device
			else type(model.rotary_emb)(config).to(norm_device)
		)
		model.rotary_emb = self._rotary_first
		model.eval().requires_grad_(False)

		self.hidden_size = int(config.hidden_size)
		self.model = model
		self._first_layers = list(model.layers[:split])
		self._second_layers = list(model.layers[split:])
		self.first_device = first
		self.last_device = norm_device
		for device in dict.fromkeys((first, norm_device)):
			if device.type == "cuda":
				torch.cuda.empty_cache()
				print(
					f"{device}: {torch.cuda.memory_allocated(device) / 2**30:.2f} GiB weights"
				)
		print(
			f"Backbone: {layer_count} layers split at {split} on {first} / {norm_device} "
			f"({precision}) in {time.perf_counter() - started:.0f}s"
		)

	@staticmethod
	def _stream_weights(
		model: nn.Module,
		merged_dir: Path,
		dtype: torch.dtype,
		first: torch.device,
		last: torch.device,
		norm_device: torch.device,
		split: int,
	) -> None:
		"""Stream safetensor slices straight to their target device without host copies."""
		index_path = merged_dir / "model.safetensors.index.json"
		if index_path.is_file():
			weight_map: dict[str, str] = json.loads(
				index_path.read_text(encoding="utf-8")
			)["weight_map"]
			by_shard: dict[str, list[str]] = {}
			for key, shard in weight_map.items():
				by_shard.setdefault(shard, []).append(key)
		else:
			by_shard = {"model.safetensors": list(model.state_dict().keys())}

		expected = set(model.state_dict().keys())
		state: dict[str, Tensor] = {}
		for shard, keys in sorted(by_shard.items()):
			with safe_open(merged_dir / shard, framework="pt", device="cpu") as handle:
				for key in keys:
					target = SplitBackbone._target_device(
						key, first, last, norm_device, split
					)
					state[key] = handle.get_tensor(key).to(device=target, dtype=dtype)
		if set(state.keys()) != expected:
			missing = sorted(expected - set(state.keys()))[:5]
			extra = sorted(set(state.keys()) - expected)[:5]
			raise ValueError(
				f"Merged weights mismatch (missing={missing}, unexpected={extra})."
			)
		model.load_state_dict(state, strict=True, assign=True)

	@staticmethod
	def _target_device(
		key: str,
		first: torch.device,
		last: torch.device,
		norm_device: torch.device,
		split: int,
	) -> torch.device:
		if key.startswith("embed_tokens."):
			return first
		if key.startswith("norm."):
			return norm_device
		if key.startswith("layers."):
			layer_index = int(key.split(".", 2)[1])
			return first if layer_index < split else last
		raise ValueError(f"Unexpected backbone parameter key: {key}")

	def encode_pooled(self, input_ids: Tensor, last_positions: Tensor) -> Tensor:
		"""Run stage 0 then stage 1 and pool the last real token before RMSNorm."""
		hidden = self.model.embed_tokens(input_ids)
		pos_first = torch.arange(hidden.shape[1], device=self.first_device).unsqueeze(0)
		cos_sin_first = self._rotary_first(hidden, pos_first)
		for layer in self._first_layers:
			hidden = self._run_layer(layer, hidden, cos_sin_first, pos_first)
		del pos_first, cos_sin_first
		if self._second_layers:
			hidden = hidden.to(self.last_device)
			pos_last = torch.arange(hidden.shape[1], device=self.last_device).unsqueeze(
				0
			)
			cos_sin_last = self._rotary_last(hidden, pos_last)
			for layer in self._second_layers:
				hidden = self._run_layer(layer, hidden, cos_sin_last, pos_last)
			del pos_last, cos_sin_last
		rows = torch.arange(hidden.shape[0], device=self.last_device)
		return self.model.norm(hidden[rows, last_positions]).float()

	@staticmethod
	def _run_layer(
		layer: nn.Module,
		hidden: Tensor,
		position_embeddings: tuple[Tensor, Tensor],
		position_ids: Tensor,
	) -> Tensor:
		out = layer(
			hidden,
			attention_mask=None,
			position_embeddings=position_embeddings,
			position_ids=position_ids,
			past_key_values=None,
			use_cache=False,
		)
		return out[0] if isinstance(out, tuple) else out


class TokenBudgetPlanner:
	"""Pack descending-length branches via best-fit vs contiguous DP."""

	def __init__(self, token_budget: int, max_rows: int) -> None:
		if token_budget <= 0 or max_rows <= 0:
			raise ValueError("token_budget and max_rows must be positive.")
		self._budget = token_budget
		self._max_rows = max_rows

	def plan(self, lengths: np.ndarray) -> list[np.ndarray]:
		order = np.argsort(-lengths, kind="stable")
		oversized: list[np.ndarray] = []
		eligible: list[int] = []
		for idx in order.tolist():
			if int(lengths[idx]) > self._budget or self._max_rows == 1:
				oversized.append(np.asarray([idx], dtype=np.int64))
			else:
				eligible.append(idx)
		if not eligible:
			return oversized
		best_fit, bf_tokens = self._best_fit(eligible, lengths)
		dp_batches = self._pack_dp(eligible, lengths, (len(best_fit), bf_tokens))
		chosen = best_fit if dp_batches is None else dp_batches
		return oversized + [np.asarray(batch, dtype=np.int64) for batch in chosen]

	def _best_fit(
		self, indices: list[int], lengths: np.ndarray
	) -> tuple[list[list[int]], int]:
		batches: list[list[int]] = []
		maximum = 0
		padded = 0
		for idx in indices:
			length = int(lengths[idx])
			if (
				not batches
				or len(batches[-1]) >= self._max_rows
				or (len(batches[-1]) + 1) * maximum > self._budget
			):
				batches.append([idx])
				maximum = length
			else:
				batches[-1].append(idx)
			padded += maximum
		return batches, padded

	def _pack_dp(
		self,
		indices: list[int],
		lengths: np.ndarray,
		incumbent: tuple[int, int],
	) -> list[list[int]] | None:
		seq = [int(lengths[idx]) for idx in indices]
		rows = len(seq)
		counts = [0] * (rows + 1)
		tokens = [0] * (rows + 1)
		cuts = [0] * rows
		budget, cap = self._budget, self._max_rows
		for start in range(rows - 1, -1, -1):
			maximum = seq[start]
			best_count = 1 + counts[start + 1]
			best_tokens = maximum + tokens[start + 1]
			best_cut = start + 1
			limit = min(rows, start + min(cap, budget // maximum))
			for cut in range(start + 2, limit + 1):
				c = 1 + counts[cut]
				t = (cut - start) * maximum + tokens[cut]
				if c < best_count or (c == best_count and t < best_tokens):
					best_count, best_tokens, best_cut = c, t, cut
			counts[start] = best_count
			tokens[start] = best_tokens
			cuts[start] = best_cut
		if (counts[0], tokens[0]) >= incumbent:
			return None
		result: list[list[int]] = []
		pos = 0
		while pos < rows:
			nxt = cuts[pos]
			result.append(indices[pos:nxt])
			pos = nxt
		return result

## Pipelined encoder

Each batch is right-padded without a mask (causal attention protects real tokens),
encoded across both stages and pooled at the last real token. Up to `pipeline_depth`
batches are enqueued before the oldest one is synchronised, so GPU0 runs batch `k+1`
while GPU1 finishes batch `k`. A CUDA out-of-memory error drains pending work, empties
caches, halves the batch and lowers the token limit; a single branch that still fails
is reported and later scored with the class prior.


In [ ]:
@dataclass(slots=True)
class InflightBatch:
	indices: np.ndarray
	features: Tensor
	event: torch.cuda.Event | None
	padded_tokens: int


class PipelinedEncoder:
	"""Encode unique branches into FP32 last-token features."""

	def __init__(
		self,
		backbone: SplitBackbone,
		attention: AttentionPolicy,
		corpus: BranchCorpus,
		depth: int,
		token_budget: int,
	) -> None:
		self._backbone = backbone
		self._attention = attention
		self._corpus = corpus
		self._depth = max(1, depth)
		self._limit = token_budget
		self._lengths = corpus.lengths
		self.oom_retries = 0
		self.failed: list[int] = []

	def encode(self, batches: list[np.ndarray]) -> np.ndarray:
		count = len(self._corpus.tokens)
		features = np.full(
			(count, self._backbone.hidden_size), np.nan, dtype=np.float32
		)
		queue = deque(batches)
		inflight: deque[InflightBatch] = deque()
		total = sum(int(self._lengths[b].max()) * len(b) for b in batches)
		started = time.perf_counter()
		with (
			tqdm(total=total, unit="tok", unit_scale=True, desc="Encoding") as bar,
			torch.inference_mode(),
		):
			while queue or inflight:
				if queue and len(inflight) < self._depth:
					batch = queue.popleft()
					if len(batch) > 1 and self._padded(batch) > self._limit:
						half = len(batch) // 2
						queue.appendleft(batch[half:])
						queue.appendleft(batch[:half])
						continue
					try:
						result = self._submit(batch)
					except torch.OutOfMemoryError:
						result = None
					if result is None:
						self._recover(batch, queue, inflight, features, bar)
					else:
						inflight.append(result)
					continue
				self._collect(inflight.popleft(), features, bar)
		elapsed = time.perf_counter() - started
		print(
			f"Encoded {total:,} padded tokens in {elapsed / 60:.1f} min "
			f"({total / max(elapsed, 1e-6):,.0f} tok/s) | OOM retries {self.oom_retries} "
			f"| failed branches {len(self.failed)}"
		)
		return features

	def _padded(self, batch: np.ndarray) -> int:
		return int(self._lengths[batch].max()) * len(batch)

	def _submit(self, batch: np.ndarray) -> InflightBatch:
		lengths = self._lengths[batch]
		padded = np.full(
			(len(batch), int(lengths.max())),
			self._corpus.pad_token_id,
			dtype=np.int64,
		)
		for row, index in enumerate(batch):
			padded[row, : lengths[row]] = self._corpus.tokens[index]
		first, last = self._backbone.first_device, self._backbone.last_device
		ids = self._to_device(padded, first)
		positions = self._to_device(lengths - 1, last)
		with self._attention.kernels():
			pooled = self._backbone.encode_pooled(ids, positions)
		del ids, positions
		if last.type != "cuda":
			return InflightBatch(batch, pooled, None, int(padded.size))
		host = torch.empty(pooled.shape, dtype=torch.float32, pin_memory=True)
		host.copy_(pooled, non_blocking=True)
		event = torch.cuda.Event()
		event.record(torch.cuda.current_stream(last))
		return InflightBatch(batch, host, event, int(padded.size))

	@staticmethod
	def _to_device(array: np.ndarray, device: torch.device) -> Tensor:
		"""Copy asynchronously from pinned memory so the host never waits on the GPU."""
		tensor = torch.from_numpy(array)
		if device.type == "cuda":
			tensor = tensor.pin_memory()
		return tensor.to(device, non_blocking=True)

	def _collect(self, item: InflightBatch, features: np.ndarray, bar: tqdm) -> None:
		if item.event is not None:
			item.event.synchronize()
		features[item.indices] = item.features.numpy()
		bar.update(item.padded_tokens)

	def _recover(self, batch, queue, inflight, features, bar) -> None:
		"""Drain in-flight work, free memory and retry the failed batch in halves."""
		while inflight:
			self._collect(inflight.popleft(), features, bar)
		gc.collect()
		for index in range(torch.cuda.device_count()):
			torch.cuda.synchronize(index)
		torch.cuda.empty_cache()
		self.oom_retries += 1
		if len(batch) == 1:
			index = int(batch[0])
			self.failed.append(index)
			bar.update(int(self._lengths[index]))
			print(
				f"\nOOM on a single branch of {int(self._lengths[index]):,} tokens; using the prior."
			)
			return
		self._limit = max(1, self._padded(batch) // 2)
		half = len(batch) // 2
		queue.appendleft(batch[half:])
		queue.appendleft(batch[:half])
		print(f"\nOOM: token limit lowered to {self._limit:,}.")

## Scoring and submission

Apply the configured temperature to the head logits before softmax:
`probabilities = softmax(logits / TEMPERATURE)`. The fallback class prior is
used directly for failed rows, without temperature scaling.


In [ ]:
class PairScorer:
	"""Run the FP32 head and temperature-scale logits into probabilities."""

	CHUNK = 8192

	def __init__(
		self,
		head: SwapEquivariantHead,
		device: torch.device,
		prior: tuple[float, float, float],
		temperature: float,
	) -> None:
		self._head = head.to(device)
		self._device = device
		self._temperature = temperature
		self._prior = np.asarray(prior, dtype=np.float64) / sum(prior)

	def score(
		self, features: np.ndarray, corpus: BranchCorpus
	) -> tuple[np.ndarray, int]:
		"""Return [rows, 3] probabilities and the number of prior-filled rows."""
		finite = np.isfinite(features).all(axis=1)
		valid = finite[corpus.a_index] & finite[corpus.b_index]
		probabilities = np.tile(self._prior, (len(valid), 1))
		rows = np.flatnonzero(valid)
		with torch.inference_mode():
			for start in range(0, len(rows), self.CHUNK):
				chunk = rows[start : start + self.CHUNK]
				h_a = torch.from_numpy(features[corpus.a_index[chunk]]).to(self._device)
				h_b = torch.from_numpy(features[corpus.b_index[chunk]]).to(self._device)
				logits = self._head(h_a, h_b).double()
				probabilities[chunk] = (
					torch.softmax(logits / self._temperature, dim=-1).cpu().numpy()
				)
		return probabilities, int((~valid).sum())


class SubmissionWriter:
	"""Write and validate `submission.csv` against the sample submission."""

	def __init__(self, config: SubmitConfig) -> None:
		self._config = config

	def write(self, frame: pd.DataFrame, probabilities: np.ndarray) -> pd.DataFrame:
		submission = pd.DataFrame(probabilities, columns=list(CLASS_COLUMNS))
		submission.insert(0, "id", frame["id"].to_numpy())
		self._validate(frame, submission)
		self._config.output_path.parent.mkdir(parents=True, exist_ok=True)
		submission.to_csv(self._config.output_path, index=False)
		return submission

	def _validate(self, frame: pd.DataFrame, submission: pd.DataFrame) -> None:
		values = submission[list(CLASS_COLUMNS)].to_numpy()
		if not np.isfinite(values).all() or not np.allclose(values.sum(axis=1), 1.0):
			raise ValueError("Probabilities must be finite and sum to one.")
		path = self._config.sample_submission
		if path.is_file():
			sample = pd.read_csv(path)
			if list(sample.columns) != list(submission.columns):
				raise ValueError(
					f"Columns differ from the sample: {list(sample.columns)}."
				)
			if not sample["id"].equals(submission["id"]):
				raise ValueError(
					"Submission ids or order differ from sample_submission.csv."
				)
		else:
			print("sample_submission.csv not found; skipped the id/column check.")


class SubmissionRunner:
	"""Orchestrate tokenization, encoding, scoring and writing."""

	def __init__(self, config: SubmitConfig) -> None:
		self._config = config

	def run(self) -> pd.DataFrame:
		started = time.perf_counter()
		config = self._config
		devices = self._devices()
		frame = pd.read_csv(config.test_csv)
		artifacts = ModelArtifacts.discover(config.model_dir, config.staging_dir)
		print(
			f"Merged backbone: {artifacts.merged_dir}\nHead checkpoint: {artifacts.adapter_path}"
		)
		tokenizer = AutoTokenizer.from_pretrained(artifacts.merged_dir)
		corpus = BranchTokenizer(tokenizer).build(frame)
		batches = TokenBudgetPlanner(
			config.token_budget, config.max_rows_per_batch
		).plan(corpus.lengths)
		padded_total = sum(int(corpus.lengths[b].max()) * len(b) for b in batches)
		real_total = int(corpus.lengths.sum())
		efficiency = 100.0 * real_total / max(1, padded_total)
		print(
			f"{len(batches):,} batches, token budget {config.token_budget:,} "
			f"(packing efficiency {efficiency:.1f}%)"
		)

		policy = AttentionPolicy(config.use_gqa, devices)
		policy.select()
		with policy:
			backbone = SplitBackbone(
				artifacts.merged_dir, devices, config.split_layer, config.precision
			)
			head = HeadLoader(artifacts).load(backbone.hidden_size)
			encoder = PipelinedEncoder(
				backbone, policy, corpus, config.pipeline_depth, config.token_budget
			)
			features = encoder.encode(batches)
		del backbone
		gc.collect()
		for index in range(torch.cuda.device_count()):
			with torch.cuda.device(index):
				torch.cuda.empty_cache()

		scorer = PairScorer(head, devices[-1], config.class_prior, config.temperature)
		probabilities, fallback_rows = scorer.score(features, corpus)
		submission = SubmissionWriter(config).write(frame, probabilities)
		print(
			f"Wrote {config.output_path} ({len(submission):,} rows) | prior-filled rows: {fallback_rows} "
			f"| temperature {config.temperature:g} "
			f"| mean probabilities {dict(zip(CLASS_COLUMNS, probabilities.mean(axis=0).round(4)))} "
			f"| total {(time.perf_counter() - started) / 60:.1f} min"
		)
		if fallback_rows:
			print(
				f"WARNING: {fallback_rows} rows had non-finite features or failed OOM retries."
			)
		return submission

	def _devices(self) -> list[torch.device]:
		requested = [torch.device(name) for name in self._config.devices]
		if all(d.type == "cuda" for d in requested):
			if torch.cuda.device_count() < max(d.index or 0 for d in requested) + 1:
				print("Fewer GPUs than configured: running on cuda:0 only (no split).")
				return [torch.device("cuda:0")]
		return requested

## Run


In [ ]:
submission = SubmissionRunner(CONFIG).run()
submission.head()